# 1. Setup

1.1 Mounting Google Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1.2 Imports and Installs

The transformers library from Hugging Face is main tool for the Vision Transformer.

In [7]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio
!pip install transformers datasets timm
!pip install scikit-learn matplotlib pandas opencv-python pillow tqdm

In [ ]:
# Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Core ML & Data Science
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

# Vision & Image Processing
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

# Hugging Face Transformers for ViT
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer

# System & Utilities
import os
from pathlib import Path
import random
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Data Acquisition & Initial Exploration

In [ ]:
# Define the correct base path
base_data_path = "/content/drive/MyDrive/Colab Notebooks/Brain_tumor_dataset"

# Check both Training and Testing directories
train_path = os.path.join(base_data_path, "Training")
test_path = os.path.join(base_data_path, "Testing")

print("Dataset Structure Analysis:")
print("=" * 50)

# Verify paths exist
print(f"Training path exists: {os.path.exists(train_path)}")
print(f"Testing path exists: {os.path.exists(test_path)}")

# Analyze Training set
print("\n TRAINING SET ANALYSIS:")
train_counts = {}
for class_name in ['glioma', 'meningioma', 'pituitary', 'notumor']:
    class_path = os.path.join(train_path, class_name)
    if os.path.exists(class_path):
        num_images = len([f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))])
        train_counts[class_name] = num_images
        print(f"  {class_name}: {num_images} images")
    else:
        print(f"  {class_name} folder not found!")

# Analyze Testing set
print("\n TESTING SET ANALYSIS:")
test_counts = {}
for class_name in ['glioma', 'meningioma', 'pituitary', 'notumor']:
    class_path = os.path.join(test_path, class_name)
    if os.path.exists(class_path):
        num_images = len([f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))])
        test_counts[class_name] = num_images
        print(f"  {class_name}: {num_images} images")
    else:
        print(f"  {class_name} folder not found!")

# Total counts
total_train = sum(train_counts.values())
total_test = sum(test_counts.values())
print(f"\n SUMMARY:")
print(f"  Total Training images: {total_train}")
print(f"  Total Testing images: {total_test}")
print(f"  Dataset Total: {total_train + total_test}")

In [ ]:
# Visualization 1: Bar chart comparing class distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Training distribution
ax1.bar(train_counts.keys(), train_counts.values(), color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
ax1.set_title('Training Set - Class Distribution')
ax1.set_ylabel('Number of Images')
ax1.tick_params(axis='x', rotation=45)

# Testing distribution
ax2.bar(test_counts.keys(), test_counts.values(), color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
ax2.set_title('Testing Set - Class Distribution')
ax2.set_ylabel('Number of Images')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Visualization 2: Pie chart for overall distribution
all_counts = {k: train_counts.get(k, 0) + test_counts.get(k, 0) for k in set(train_counts) | set(test_counts)}
plt.figure(figsize=(8, 8))
plt.pie(all_counts.values(), labels=all_counts.keys(), autopct='%1.1f%%',
        colors=['#ff9999','#66b3ff','#99ff99','#ffcc99'])
plt.title('Overall Class Distribution in Entire Dataset')
plt.show()

In [ ]:
# Display sample images from each class
def display_sample_images(data_path, title):
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    fig.suptitle(title, fontsize=16)

    classes = ['glioma', 'meningioma', 'pituitary', 'notumor']

    for col, class_name in enumerate(classes):
        class_path = os.path.join(data_path, class_name)
        if os.path.exists(class_path):
            # Get random samples
            images = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
            if len(images) >= 2:
                sample1, sample2 = random.sample(images, 2)

                # Display first sample
                img1 = Image.open(os.path.join(class_path, sample1))
                axes[0, col].imshow(img1)
                axes[0, col].set_title(f'{class_name}\n\nSample 1')
                axes[0, col].axis('off')

                # Display second sample
                img2 = Image.open(os.path.join(class_path, sample2))
                axes[1, col].imshow(img2)
                axes[1, col].set_title(f'{class_name}\n\nSample 2')
                axes[1, col].axis('off')

    plt.tight_layout()
    plt.show()

# Display samples from training set
display_sample_images(train_path, "Sample Images from Training Set")

# Analyze image dimensions and properties
print("\n ANALYZING IMAGE PROPERTIES:")
all_widths = []
all_heights = []
formats = []

# Check sample of images from each class
for class_name in ['glioma', 'meningioma', 'pituitary', 'notumor']:
    class_path = os.path.join(train_path, class_name)
    if os.path.exists(class_path):
        images = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
        if images:
            # Check first 10 images from each class
            for img_name in images[:10]:
                img_path = os.path.join(class_path, img_name)
                with Image.open(img_path) as img:
                    all_widths.append(img.width)
                    all_heights.append(img.height)
                    formats.append(img.format if img.format else 'Unknown')

print(f"Image dimensions range: {min(all_widths)}x{min(all_heights)} to {max(all_widths)}x{max(all_heights)}")
print(f"Average dimensions: {np.mean(all_widths):.1f}x{np.mean(all_heights):.1f}")
print(f"Unique formats: {set(formats)}")

# Plot dimension distribution
plt.figure(figsize=(10, 5))
plt.scatter(all_widths, all_heights, alpha=0.6)
plt.xlabel('Width (pixels)')
plt.ylabel('Height (pixels)')
plt.title('Image Dimensions Scatter Plot')
plt.grid(True, alpha=0.3)
plt.show()

# Data Preprocessing & Train/Val/Test Split

Data Pipeline

In [ ]:
print("PHASE 3: Data Preprocessing & Splitting")
print("=" * 50)

# First, let's create a proper dataset structure
class BrainMRIDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.label_to_idx = {'glioma': 0, 'meningioma': 1, 'pituitary': 2, 'notumor': 3}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]

        # Load image
        image = Image.open(image_path).convert('RGB')  # Ensure RGB format

        if self.transform:
            image = self.transform(image)

        label_idx = self.label_to_idx[label]
        return image, label_idx

# Collect all training image paths and labels
print("📁 Collecting all training images...")
all_train_images = []
all_train_labels = []

for class_name in ['glioma', 'meningioma', 'pituitary', 'notumor']:
    class_path = os.path.join(train_path, class_name)
    if os.path.exists(class_path):
        images = [os.path.join(class_path, f) for f in os.listdir(class_path)
                 if f.endswith(('.jpg', '.png', '.jpeg'))]
        all_train_images.extend(images)
        all_train_labels.extend([class_name] * len(images))
        print(f"  {class_name}: {len(images)} images")

print(f"Total training images collected: {len(all_train_images)}")

# Perform the critical 80/20 stratified split
print("\n🎯 Performing stratified 80/20 train/validation split...")
train_images, val_images, train_labels, val_labels = train_test_split(
    all_train_images, all_train_labels,
    test_size=0.2,
    random_state=42,
    stratify=all_train_labels  # This preserves class distribution in both splits
)

print(f"New Training set: {len(train_images)} images")
print(f"Validation set: {len(val_images)} images")

# Verify the stratification worked
print("\n✅ Verifying class distribution in splits:")
def check_class_distribution(images, labels, split_name):
    print(f"{split_name} class distribution:")
    for class_name in ['glioma', 'meningioma', 'pituitary', 'notumor']:
        count = sum(1 for label in labels if label == class_name)
        percentage = (count / len(labels)) * 100
        print(f"  {class_name}: {count} images ({percentage:.1f}%)")

check_class_distribution(train_images, train_labels, "Training")
check_class_distribution(val_images, val_labels, "Validation")

ViT-specific preprocessing and data loaders

In [ ]:
# ViT-specific preprocessing
print("\n🔧 Setting up ViT preprocessing...")

# Load ViT processor - this handles all the necessary transformations
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')

# Define transforms for training (with augmentation) and validation (without augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

# Test transform (same as validation)
test_transform = val_transform

print("ViT preprocessing parameters:")
print(f"  Input size: 224x224")
print(f"  Normalization: mean={processor.image_mean}, std={processor.image_std}")

# Create datasets
print("\n📊 Creating PyTorch datasets...")
train_dataset = BrainMRIDataset(train_images, train_labels, transform=train_transform)
val_dataset = BrainMRIDataset(val_images, val_labels, transform=val_transform)

# For test set, we'll collect images from the original Testing folder
print("📁 Collecting test images...")
test_images = []
test_labels = []

for class_name in ['glioma', 'meningioma', 'pituitary', 'notumor']:
    class_path = os.path.join(test_path, class_name)
    if os.path.exists(class_path):
        images = [os.path.join(class_path, f) for f in os.listdir(class_path)
                 if f.endswith(('.jpg', '.png', '.jpeg'))]
        test_images.extend(images)
        test_labels.extend([class_name] * len(images))

test_dataset = BrainMRIDataset(test_images, test_labels, transform=test_transform)

print(f"Test set: {len(test_images)} images")
check_class_distribution(test_images, test_labels, "Test")

# Create data loaders
batch_size = 128  # Adjust based on your Colab GPU memory

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"\n✅ Data pipeline ready!")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")
print(f"   Batch size: {batch_size}")

# Let's verify one batch works correctly
print("\n🔍 Verifying data pipeline with one batch...")
for images, labels in train_loader:
    print(f"Batch image shape: {images.shape}")  # Should be [batch_size, 3, 224, 224]
    print(f"Batch label shape: {labels.shape}")  # Should be [batch_size]
    print(f"Image range: [{images.min():.3f}, {images.max():.3f}]")
    break

# Model Building & Training

Load Pre-trained ViT Model

In [ ]:
print("🔧 Loading Pre-trained Vision Transformer...")

model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224-in21k',
    num_labels=4,
    ignore_mismatched_sizes=True
)

print("✅ ViT Model loaded successfully!")
print(f"Model architecture: {model.__class__.__name__}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

Calculate Class Weights

In [ ]:
print("📊 Calculating class weights for imbalance...")

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Convert string labels to numerical indices for weight calculation
label_to_idx = {'glioma': 0, 'meningioma': 1, 'pituitary': 2, 'notumor': 3}
train_labels_idx = [label_to_idx[label] for label in train_labels]

# Calculate class weights
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_labels_idx),
    y=train_labels_idx
)

class_weights = torch.tensor(class_weights, dtype=torch.float32)
print(f"Class weights: {class_weights}")

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}")

Setup Training Configuration

In [ ]:
print("⚙️ Setting up training configuration...")

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# Training parameters for speed
epochs = 5  # Reduced for faster training
learning_rate = 5e-4  # Slightly higher for faster convergence

# Optimizer and loss function with class weights
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))
# scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

print(f"Training configuration:")
print(f"  Epochs: {epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Optimizer: AdamW")
print(f"  Scheduler: CosineAnnealing")

Training Loop Function

In [ ]:
print("🏃 Setting up training loop...")

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    progress_bar = tqdm(dataloader, desc='Training')
    for batch_idx, (images, labels) in enumerate(progress_bar):
        images, labels = images.to(device), labels.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images).logits
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == labels).sum().item()
        total_samples += labels.size(0)

        # Update progress bar
        progress_bar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Acc': f'{correct_predictions/total_samples:.4f}'
        })

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct_predictions / total_samples
    return epoch_loss, epoch_acc

Validation Loop Function

In [ ]:
def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images).logits
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct_predictions / total_samples
    return epoch_loss, epoch_acc

Main Training Execution

In [ ]:
print("🚀 Starting Training...")

# Track metrics
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0.0
best_model_state = None

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-" * 40)

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # Validate
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Update scheduler
    #scheduler.step()

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        print(f"🔥 New best model saved! Val Acc: {val_acc:.4f}")

    print(f"Current LR: {scheduler.get_last_lr()[0]:.2e}")

print(f"\n✅ Training completed! Best validation accuracy: {best_val_acc:.4f}")

# Load best model for final evaluation
model.load_state_dict(best_model_state)

# Final Model Evaluation

In [ ]:
print("📊 Preparing for Final Evaluation...")

def evaluate_model(model, test_loader, device, label_names):
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Testing'):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images).logits
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            _, predictions = torch.max(outputs, 1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    return all_labels, all_predictions, all_probabilities

# Label mapping for readable results
label_names = ['glioma', 'meningioma', 'pituitary', 'notumor']
idx_to_label = {0: 'glioma', 1: 'meningioma', 2: 'pituitary', 3: 'notumor'}

print("✅ Evaluation functions ready!")

# Results

In [ ]:
print("📈 Setting up visualization tools...")

def plot_training_history(train_losses, val_losses, train_accs, val_accs):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plot losses
    ax1.plot(train_losses, label='Training Loss', color='blue', linewidth=2)
    ax1.plot(val_losses, label='Validation Loss', color='red', linewidth=2)
    ax1.set_title('Training and Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot accuracies
    ax2.plot(train_accs, label='Training Accuracy', color='blue', linewidth=2)
    ax2.plot(val_accs, label='Validation Accuracy', color='red', linewidth=2)
    ax2.set_title('Training and Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

    plt.figure(figsize=(8, 6))
    disp.plot(cmap='Blues', values_format='d')
    plt.title('Confusion Matrix - Test Set')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return cm

print("✅ Visualization tools ready!")

# Results & Analysis

In [ ]:
print("🎯 Setting up comprehensive performance metrics...")

def calculate_detailed_metrics(y_true, y_pred, y_probs, class_names):
    print("=" * 60)
    print("COMPREHENSIVE PERFORMANCE REPORT")
    print("=" * 60)

    # Overall accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print(f"\n📊 Overall Test Accuracy: {accuracy:.4f}")

    # Detailed classification report
    print("\n📋 Detailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # Per-class metrics
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None)

    print("\n🎯 Per-Class Performance:")
    metrics_df = pd.DataFrame({
        'Class': class_names,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Support': support
    })
    print(metrics_df.round(4))

    return accuracy, metrics_df

def plot_class_performance(metrics_df):
    plt.figure(figsize=(12, 6))

    x = range(len(metrics_df))
    width = 0.25

    plt.bar([i - width for i in x], metrics_df['Precision'], width, label='Precision', alpha=0.8)
    plt.bar(x, metrics_df['Recall'], width, label='Recall', alpha=0.8)
    plt.bar([i + width for i in x], metrics_df['F1-Score'], width, label='F1-Score', alpha=0.8)

    plt.xlabel('Classes')
    plt.ylabel('Score')
    plt.title('Per-Class Performance Metrics')
    plt.xticks(x, metrics_df['Class'])
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

print("✅ Performance metrics setup complete!")

Final Evaluation Execution (Run after training completes)

In [ ]:
print("🎯 RUNNING FINAL EVALUATION ON TEST SET...")
print("=" * 50)

# Record start time
import time
start_time = time.time()

# Evaluate on test set
y_true, y_pred, y_probs = evaluate_model(model, test_loader, device, label_names)

# Calculate metrics
test_accuracy, metrics_df = calculate_detailed_metrics(y_true, y_pred, y_probs, label_names)

# Plot results
plot_training_history(train_losses, val_losses, train_accs, val_accs)
cm = plot_confusion_matrix(y_true, y_pred, label_names)
plot_class_performance(metrics_df)

# Calculate execution time
end_time = time.time()
training_time = (end_time - start_time) / 60  # in minutes

print(f"\n⏱️  Total evaluation time: {training_time:.2f} minutes")

# Create results summary for group comparison
vit_results = create_results_summary(
    model_name='Vision Transformer (ViT)',
    test_accuracy=test_accuracy,
    precision=metrics_df['Precision'].values,
    recall=metrics_df['Recall'].values,
    f1=metrics_df['F1-Score'].values,
    training_time=training_time
)

print("\n📋 ViT Results Summary for Group Comparison:")
for key, value in vit_results.items():
    print(f"  {key}: {value}")